# PolySight: reproducción de entrenamiento en Google Colab

Este notebook repite `main16-baseline` con semilla 42 mediante el paquete versionado de PolySight. Conserva el protocolo de CEDIA y compara las métricas, pero no promete checkpoints idénticos bit a bit entre GPU y versiones de CUDA/cuDNN diferentes. No constituye validación clínica.

## Requisitos previos

1. Seleccionar un runtime GPU con Python 3.11.
2. Guardar en Google Drive el ZIP canónico de HyperKvasir.
3. Guardar los pesos iniciales `efficientnet_b0_rwightman-7f5810bc.pth`.
4. Ajustar las rutas y la URL del repositorio en la siguiente celda.

No escribas tokens, contraseñas ni datos clínicos identificables en el notebook.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ORGANIZACION/polysight.git"
GIT_REF = "87ac0694dc8e53f6d295b902ce4465af8dfd2aff"
WORKSPACE = Path("/content/polysight-training")
WORK_ROOT = Path("/content/polysight-work")
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/polysight-inputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/polysight-colab-results")
DATASET_ARCHIVE = DRIVE_INPUT_DIR / "hyper-kvasir-labeled-images.zip"
INITIAL_WEIGHTS_PATH = (
    DRIVE_INPUT_DIR / "efficientnet_b0_rwightman-7f5810bc.pth"
)
RUN_TEST_EVALUATION = False
EXPECTED_DATASET_SIZE = 3_928_814_344
EXPECTED_DATASET_SHA256 = (
    "c603449b1bc0be86948b11d9aea8b2002058a11e6f5499e2a384b9ae9c8dbd3f"
)
EXPECTED_MANIFEST_SHA256 = (
    "8f59d5c5f1d188ad75d1b28c6ab9b56e3b3c68bcf8f7dd0940422dc8df27f463"
)
EXPECTED_WEIGHTS_SHA256_PREFIX = "7f5810bc"

## 1. Verificar Python y montar Google Drive

El release fija Python 3.11. Drive conserva entradas y resultados más allá de la sesión temporal de Colab.

In [ ]:
import platform
import sys
from google.colab import drive

print("Python:", platform.python_version())
print("Ejecutable:", sys.executable)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Este release requiere un runtime de Python 3.11")
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Obtener e instalar el código exacto

Se instala el paquete real. El notebook no contiene una segunda implementación del entrenamiento.

In [ ]:
import subprocess

def run(command: list[str], cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if not WORKSPACE.exists():
    run(["git", "clone", REPO_URL, str(WORKSPACE)])
run(["git", "checkout", GIT_REF], cwd=WORKSPACE)
run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=WORKSPACE)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=WORKSPACE, text=True
).strip()
print("Commit:", commit)

## 3. Diagnosticar GPU y registrar versiones

El entrenamiento de PolySight exige CUDA. La identidad del hardware y las versiones se guardarán también como tags del run.

In [ ]:
import json
import mlflow
import torch
import torchvision

environment_report = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "mlflow": mlflow.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "git_commit": commit,
}
print(json.dumps(environment_report, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("Selecciona un runtime GPU antes de entrenar")

## 4. Verificar dataset y pesos iniciales

El ZIP debe coincidir por tamaño y SHA-256. El prefijo de hash de los pesos forma parte de su nombre oficial; el hash completo observado se registrará en los resultados.

In [ ]:
import hashlib

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for required_path in (DATASET_ARCHIVE, INITIAL_WEIGHTS_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Falta el archivo requerido: {required_path}")
if DATASET_ARCHIVE.stat().st_size != EXPECTED_DATASET_SIZE:
    raise ValueError("El tamaño del ZIP de HyperKvasir no coincide")
dataset_sha256 = sha256_file(DATASET_ARCHIVE)
weights_sha256 = sha256_file(INITIAL_WEIGHTS_PATH)
print("Dataset SHA-256:", dataset_sha256)
print("Pesos SHA-256:", weights_sha256)
if dataset_sha256 != EXPECTED_DATASET_SHA256:
    raise ValueError("El ZIP no es el dataset canónico utilizado por PolySight")
if not weights_sha256.startswith(EXPECTED_WEIGHTS_SHA256_PREFIX):
    raise ValueError("Los pesos iniciales no tienen el hash esperado")

## 5. Preparar datos y regenerar el split

La preparación es idempotente. El entrenamiento solo continúa si el manifest generado coincide con el hash auditado de `main16`.

In [ ]:
DATA_PARENT = WORK_ROOT / "data" / "hyper-kvasir"
DATA_DIR = DATA_PARENT / "labeled-images"
MANIFEST_ROOT = WORK_ROOT / "manifests"
MANIFEST_PATH = MANIFEST_ROOT / "main16" / "manifest.csv"
RUNS_DIR = WORK_ROOT / "runs"
MLRUNS_DIR = WORK_ROOT / "mlruns"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
run(
    ["polysight-prepare", "--archive", str(DATASET_ARCHIVE), "--output-dir", str(DATA_PARENT)],
    cwd=WORKSPACE,
)
run(
    ["polysight-split", "--data-dir", str(DATA_PARENT), "--output-dir", str(MANIFEST_ROOT), "--profile", "main16"],
    cwd=WORKSPACE,
)
manifest_sha256 = sha256_file(MANIFEST_PATH)
print("Manifest SHA-256:", manifest_sha256)
if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise ValueError("El split generado no coincide con el manifest auditado")

## 6. Configurar rutas y MLflow

MLflow usa un almacén local dentro de `/content`; al final se copia a Drive junto con checkpoints y métricas.

In [ ]:
import os

training_env = os.environ.copy()
training_env.update(
    {
        "POLYSIGHT_DATA_DIR": str(DATA_DIR),
        "POLYSIGHT_MANIFEST_DIR": str(MANIFEST_ROOT),
        "POLYSIGHT_WEIGHTS_PATH": str(INITIAL_WEIGHTS_PATH),
        "POLYSIGHT_TRACKING_URI": f"file:{MLRUNS_DIR}",
        "POLYSIGHT_RUNS_DIR": str(RUNS_DIR),
    }
)
print(json.dumps({key: training_env[key] for key in training_env if key.startswith("POLYSIGHT_")}, indent=2))

## 7. Smoke test obligatorio

Este entrenamiento corto comprueba carga de datos, augmentations, forward, backward, AMP, checkpoint y tracking antes de consumir el presupuesto completo.

In [ ]:
run(
    ["polysight-train", "--config", "configs/smoke-main16.yaml", "--seed", "42"],
    cwd=WORKSPACE,
    env=training_env,
)

## 8. Repetir `main16-baseline`, semilla 42

La configuración conserva 3 épocas de cabeza, hasta 30 de fine-tuning, paciencia 7, batch 128 y AMP. No cambies estos valores si el objetivo es comparar con el experimento original.

In [ ]:
run(
    ["polysight-train", "--config", "configs/main16-baseline.yaml", "--seed", "42"],
    cwd=WORKSPACE,
    env=training_env,
)
checkpoint_candidates = list((RUNS_DIR / "main16-baseline").glob("*/best.pt"))
if len(checkpoint_candidates) != 1:
    raise RuntimeError(f"Se esperaba un checkpoint principal; encontrados: {checkpoint_candidates}")
BEST_CHECKPOINT = checkpoint_candidates[0]
print("Checkpoint reproducido:", BEST_CHECKPOINT)
print("SHA-256:", sha256_file(BEST_CHECKPOINT))

## 9. Completar trazabilidad y comparar validation

Las diferencias de GPU, drivers y kernels pueden cambiar el resultado. Se compara el protocolo y la cercanía de métricas, no el hash del checkpoint.

In [ ]:
from mlflow import MlflowClient

checkpoint_data = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
run_id = checkpoint_data["mlflow_run_id"]
tracking_uri = training_env["POLYSIGHT_TRACKING_URI"]
client = MlflowClient(tracking_uri=tracking_uri)
extra_tags = {
    **{f"runtime.{key}": str(value) for key, value in environment_report.items()},
    "dataset_sha256": dataset_sha256,
    "initial_weights_sha256": weights_sha256,
    "reproduction_platform": "google-colab",
}
for key, value in extra_tags.items():
    client.set_tag(run_id, key, value)
validation_path = BEST_CHECKPOINT.parent / "validation" / "metrics.json"
validation_metrics = json.loads(validation_path.read_text())
expected_validation = {
    "accuracy": 0.9159770846594526,
    "balanced_accuracy": 0.8590389720513687,
    "macro_f1": 0.8583948872106766,
    "top_3_accuracy": 0.9910884786760026,
    "weighted_f1": 0.9142967276266037,
}
comparison = {
    key: {
        "cedia": expected_validation[key],
        "colab": validation_metrics[key],
        "difference": validation_metrics[key] - expected_validation[key],
    }
    for key in expected_validation
}
print(json.dumps(comparison, indent=2))

## 10. Evaluación opcional y única sobre test

Déjala desactivada mientras investigas errores o ajustas el entorno. Solo debe habilitarse después de cerrar todas las decisiones, y sus resultados no deben provocar cambios posteriores.

In [ ]:
TEST_OUTPUT_DIR = WORK_ROOT / "test-evaluation" / "main16-baseline-seed42"
if RUN_TEST_EVALUATION:
    run(
        [
            "polysight-evaluate",
            "--config",
            "configs/main16-baseline.yaml",
            "--checkpoint",
            str(BEST_CHECKPOINT),
            "--split",
            "test",
            "--output-dir",
            str(TEST_OUTPUT_DIR),
        ],
        cwd=WORKSPACE,
        env=training_env,
    )
else:
    print("Test permanece cerrado: RUN_TEST_EVALUATION=False")

## 11. Conservar resultados en Drive

La sesión de Colab es temporal. Este paso empaqueta runs, MLflow, manifests, evaluación opcional y el reporte del entorno.

In [ ]:
import shutil
from datetime import datetime, timezone

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "environment": environment_report,
    "dataset_sha256": dataset_sha256,
    "manifest_sha256": manifest_sha256,
    "initial_weights_sha256": weights_sha256,
    "checkpoint_sha256": sha256_file(BEST_CHECKPOINT),
    "mlflow_run_id": run_id,
    "validation_comparison": comparison,
    "test_evaluated": RUN_TEST_EVALUATION,
}
report_path = WORK_ROOT / "reproduction-report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")
archive_base = DRIVE_OUTPUT_DIR / f"polysight-main16-seed42-{run_id}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=WORK_ROOT)
print("Resultados guardados en:", archive_path)

## Criterio de aceptación

La reproducción es válida cuando usa el commit, dataset, manifest, pesos iniciales, configuración y semilla documentados; completa smoke y entrenamiento; conserva trazabilidad; y obtiene métricas comparables. Un hash de checkpoint diferente no implica por sí solo que el experimento sea falso, porque CEDIA y Colab pueden usar hardware y kernels no deterministas distintos.